In [2]:
import pandas as pd

In [3]:
import numpy as np

In [4]:
subscribers= pd.read_csv("C:/Users/humak/subscribers.csv")
titles= pd.read_csv("C:/Users/humak/titles.csv")
reviews= pd.read_csv("C:/Users/humak/reviews.csv")
ratings= pd.read_csv("C:/Users/humak/ratings.csv")
watchlist= pd.read_csv("C:/Users/humak/watchlist.csv")
watch_history= pd.read_csv("C:/Users/humak/watch_history.csv")

KPI 1: Total Watch Hours

Business Meaning

How many hours were watched on the platform?

Formula

SUM(watch_duration_min) / 60

In [5]:
total_watch_hours = watch_history['watch_duration_min'].sum() / 60

print(f"Total Watch Hours: {total_watch_hours:,.2f}")

Total Watch Hours: 3,333,467.73


KPI 2. Active Subscribers

A subscriber is considered active if they watched at least one title.

In [6]:
active_subscribers = watch_history['subscriber_id'].nunique()

print(f"Active Subscribers: {active_subscribers}")

Active Subscribers: 14998


KPI 3. Active Rate

Formula
(Active Subscribers / Total Subscribers) × 100

In [8]:
total_subscribers = subscribers['subscriber_id'].nunique()

active_rate = (active_subscribers / total_subscribers) * 100

print(f"Active Rate: {active_rate:.2f}%")

Active Rate: 99.99%


KPI 4. Churn Rate

Inactive subscribers are those who never watched anything.

In [9]:
inactive_subscribers = total_subscribers - active_subscribers

churn_rate = (inactive_subscribers / total_subscribers) * 100

print(f"Churn Rate: {churn_rate:.2f}%")


Churn Rate: 0.01%


KPI 5. Average Completion Rate

Formula

Average(completion_pct)

In [10]:
avg_completion = watch_history['completion_pct'].mean()

print(f"Average Completion Rate: {avg_completion:.2f}%")

Average Completion Rate: 65.31%


KPI 6. Monthly Recurring Revenue (MRR)

Step 1

Find active subscriber IDs.

In [11]:
active_ids = watch_history['subscriber_id'].unique()

Step 2

Filter subscribers.

In [12]:
active_subs = subscribers[
    subscribers['subscriber_id'].isin(active_ids)
]

Step 3

Calculate revenue.

In [13]:
mrr = active_subs['monthly_price_usd'].sum()

print(f"Monthly Recurring Revenue: ${mrr:,.2f}")

Monthly Recurring Revenue: $227,741.02


KPI 7. ARPU (Average Revenue Per User)

Formula

MRR / Active Subscribers

In [14]:
arpu = mrr / active_subscribers

print(f"ARPU: ${arpu:.2f}")


ARPU: $15.18


KPI 8. Average Watch Time per Subscriber

Formula

Total Watch Hours / Active Subscribers

In [15]:
avg_watch_time = total_watch_hours / active_subscribers

print(f"Average Watch Time per Subscriber: {avg_watch_time:.2f} hours")

Average Watch Time per Subscriber: 222.26 hours


KPI 9. Watchlist Conversion

Step 1

Create a common key.

In [16]:
watchlist['key'] = (
    watchlist['subscriber_id']
    + "_"
    + watchlist['title_id']
)

watch_history['key'] = (
    watch_history['subscriber_id']
    + "_"
    + watch_history['title_id']
)

Step 2

Count converted entries.

In [17]:
watched = watchlist['key'].isin(watch_history['key']).sum()

watchlist_entries = len(watchlist)

Step 3

Calculate conversion.

In [18]:
conversion = (watched / watchlist_entries) * 100

print(f"Watchlist Conversion: {conversion:.2f}%")

Watchlist Conversion: 1.44%


KPI 10. Hit Concentration

Count plays.

In [19]:
plays = (
    watch_history
    .groupby('title_id')
    .size()
    .sort_values(ascending=False)
)

Top 10% of titles.

In [22]:
top_titles = int(len(plays) * 0.10)

top_plays = plays.head(top_titles).sum()

total_plays = plays.sum()

KPI

In [23]:
hit_concentration = (top_plays / total_plays) * 100

print(f"Hit Concentration: {hit_concentration:.2f}%")

Hit Concentration: 30.99%


KPI 11. Originals Share of Watch Hours

Merge tables.

In [24]:
merged = watch_history.merge(
    titles[['title_id', 'is_original']],
    on='title_id',
    how='left'
)


Calculate hours.

In [25]:
original_hours = (
    merged.loc[merged['is_original'], 'watch_duration_min']
    .sum() / 60
)

total_hours = merged['watch_duration_min'].sum() / 60

original_share = (original_hours / total_hours) * 100

print(f"Originals Share of Hours: {original_share:.2f}%")

Originals Share of Hours: 27.17%


⭐ Bonus KPI – Cohort Retention

Step 1: Create signup cohorts

In [26]:
# 1. Convert dates
subscribers['signup_date'] = pd.to_datetime(subscribers['signup_date'])
watch_history['watch_date'] = pd.to_datetime(watch_history['watch_date'])

# 2. Create cohort month
subscribers['cohort_month'] = subscribers['signup_date'].dt.to_period('M')

# 3. Merge subscriber information with watch activity
activity = subscribers[
    ['subscriber_id', 'signup_date', 'cohort_month']
].merge(
    watch_history[['subscriber_id', 'watch_date']],
    on='subscriber_id',
    how='left'
)

# 4. Calculate months since signup
activity['months_since_signup'] = (
    (activity['watch_date'].dt.year - activity['signup_date'].dt.year) * 12
    + 
    (activity['watch_date'].dt.month - activity['signup_date'].dt.month)
)

# 5. Keep 3, 6 and 12 month activity
retention_activity = activity[
    activity['months_since_signup'].isin([3, 6, 12])
]

# 6. Count unique active subscribers
retention = (
    retention_activity
    .groupby(['cohort_month', 'months_since_signup'])['subscriber_id']
    .nunique()
    .unstack(fill_value=0)
)

# 7. Cohort size
cohort_size = subscribers.groupby('cohort_month')['subscriber_id'].nunique()

# 8. Calculate retention %
retention_pct = retention.div(cohort_size, axis=0) * 100

print(retention_pct)

months_since_signup        3.0   6.0       12.0
cohort_month                                   
2016-01                     NaN   NaN       NaN
2016-02                     NaN   NaN       NaN
2016-03                0.000000   0.0  8.333333
2016-04                     NaN   NaN       NaN
2016-05                0.000000   0.0  5.000000
...                         ...   ...       ...
2026-01               98.706897   0.0  0.000000
2026-02              100.000000   0.0  0.000000
2026-03                     NaN   NaN       NaN
2026-04                     NaN   NaN       NaN
2026-05                     NaN   NaN       NaN

[125 rows x 3 columns]
